### Fixing the environment

In [1]:
#Importing packages
import sys
sys.path.append('/home/onyxia/work/nlp_central_banks/lyna_work')

from module import *

/home/onyxia/work/venv_gpu/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
#Importing torch and making sure the GPU is available
import sys
print(sys.executable)  # doit afficher .../venv_gpu/bin/python

import torch
print(torch.cuda.is_available())  # doit afficher True
print(torch.cuda.get_device_name(0))  # NVIDIA A2

/home/onyxia/work/venv_gpu/bin/python
True
Tesla T4


In [3]:
# ── Verification GPU ────────────────────────────────────
print(f"PyTorch  : {torch.__version__}")
print(f"CUDA     : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU      : {torch.cuda.get_device_name(0)}")
    print(f"VRAM     : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


PyTorch  : 2.6.0+cu124
CUDA     : True
GPU      : Tesla T4
VRAM     : 15.6 GB


### BERTopic pipeline
#### Preprocessing the data

I import my preprocessed data.

In [4]:
# Importing the preprocessed data from the first notebook (saved in the SSP Cloud) :
fs = s3fs.S3FileSystem(
    client_kwargs={"endpoint_url": "https://minio.lab.sspcloud.fr"}
)
fs.invalidate_cache()
with fs.open("lelkamel/chunks/df_filtered.csv", "rb") as f:
    df_filtered = pd.read_csv(f)
print(f"df_filtered chargé : {df_filtered.shape}")

df_filtered chargé : (2286, 19)


Now I will compare my text's size to that of the model that interests me, to see whether it fits the context window.

I begin by retrieving the context window size of the model that I will use on the HuggingFace.

In [11]:
from transformers import AutoConfig

config = AutoConfig.from_pretrained("hugom123/bge-centralbank")
print(config.max_position_embeddings)

512


In [12]:
df_filtered["text.len"] = df_filtered["text"].apply(len)
df_filtered["text.len"].describe()

count      2286.000000
mean      20002.870954
std       11961.596277
min        1314.000000
25%       11670.250000
50%       17653.500000
75%       25741.750000
max      122759.000000
Name: text.len, dtype: float64

My text clearly does not fit the context window of this model. To resolve this, I decide to chunk it. This strategy will also enable me to retrive topics within speeches. 

In [ ]:
# I do the chunking, thanks to a function I built, available in the module section
#if torch.cuda.is_available():
#    spacy.prefer_gpu()  # utilise GPU si CuPy dispo, sinon CPU sans erreur

#nlp = spacy.load("en_core_web_sm")

#tokenizer = AutoTokenizer.from_pretrained("hugom123/bge-centralbank")

#df_chunks = process_corpus(df_filtered)

#print(f"\nDiscours        : {len(df_filtered)}")
#print(f"Chunks          : {len(df_chunks)}")
#print(f"Chunks/discours : {len(df_chunks)/len(df_filtered):.1f}")
#print(f"\nTaille chunks (mots) :")
#print(df_chunks['n_words'].describe().round(0))
#print(f"\nChunks < 50 mots : {(df_chunks['n_words'] < 50).sum()}")
#I save it 
#df_chunks.to_csv("s3://lelkamel/chunks/df_chunks_all.csv", index=False)


Segmentation en phrases (spaCy)...


spaCy: 100%|██████████| 2286/2286 [13:53<00:00,  2.74it/s]


Chunking...


Chunking: 100%|██████████| 2286/2286 [00:26<00:00, 86.06it/s] 



Discours        : 2286
Chunks          : 23913
Chunks/discours : 10.5

Taille chunks (mots) :
count    23913.0
mean       342.0
std         58.0
min         18.0
25%        339.0
50%        359.0
75%        372.0
max        477.0
Name: n_words, dtype: float64

Chunks < 50 mots : 42


In [5]:
#To save computing time, I import here the chunk database, obtained and saved in the previous code cell.
with fs.open("lelkamel/chunks/df_chunks_all.csv", "rb") as f:
    df_chunks_all = pd.read_csv(f)

print(df_chunks_all.columns.tolist())
print(f"\nNombre de discours uniques : {df_chunks_all['doc_id'].nunique()}")
print(f"Chunks par discours (moy.) : {len(df_chunks_all)/df_chunks_all['doc_id'].nunique():.1f}")
print(f"\nAnnées : {df_chunks_all['Year'].min()} - {df_chunks_all['Year'].max()}")

['doc_id', 'chunk_id', 'CentralBank', 'Date', 'Year', 'Authorname', 'Role', 'Source', 'chunk_text', 'n_words']

Nombre de discours uniques : 2286
Chunks par discours (moy.) : 10.5

Années : 2001 - 2023


### I now create a BERTopic object, fit and transform

To create my `topic_model`, I start by creating a `BERTopic` object and define some parameters. 
- I begin by computing the embeddings of my chunks with `hugom123/bge-centralbank` and saving it in the onyxia cloud. 
- Secondly, I define my vectoriser model in order to remove common english stopwords as well as other economic and institutional stopwords to retrive meaningful topics. 
- Then, I do not change the default parameters of the clutering model at start (`hdbscan_model`), nor the dimension reduction model (`umap_model`). 
- Finally, I use the fit methode to fit the topic model to the corpus.

In [ ]:
# In this cell, I compute the embedding for all chunks
 #I begin by importing chunks, then import the model, then apply it on my data, then save it.   


#docs = df_chunks_all['chunk_text'].tolist()
#print(f"Nombre de chunks : {len(docs)}")
#print("\nChargement du modèle...")
#embedding_model = SentenceTransformer("hugom123/bge-centralbank")
#print("Calcul des embeddings...")
#embeddings = embedding_model.encode(
#    docs,
#    batch_size=64,
#    show_progress_bar=True,
#    device="cuda" if torch.cuda.is_available() else "cpu"
#)
#print(f"Embeddings shape : {embeddings.shape}")
#print("\nSauvegarde embeddings sur S3...")
#with fs.open("lelkamel/chunks/embeddings_all.npy", "wb") as f:
#    np.save(f, embeddings)
#print("Embeddings sauvegardés : s3://lelkamel/chunks/embeddings_all.npy")

Fichiers disponibles :
['lelkamel/chunks/df_all_clean.csv', 'lelkamel/chunks/df_chunks.csv', 'lelkamel/chunks/df_chunks_all.csv', 'lelkamel/chunks/df_stratified_clean.csv', 'lelkamel/chunks/embeddings.npy']

Chargement df_chunks...
df_chunks chargé : (23913, 10)
Nombre de chunks : 23913

Chargement du modèle...


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 2917.10it/s]


Calcul des embeddings...


Batches: 100%|██████████| 374/374 [33:56<00:00,  5.45s/it]


Embeddings shape : (23913, 1024)

Sauvegarde embeddings sur S3...
Embeddings sauvegardés : s3://lelkamel/chunks/embeddings_all.npy


In [6]:
#To save computing time, I import here the embeddings obtained and saved in the previous code cell.
with fs.open("lelkamel/chunks/embeddings_all.npy", "rb") as f:
    embeddings = np.load(f)

print(f"Embeddings chargés : {embeddings.shape}")

Embeddings chargés : (23913, 1024)


In [7]:
vectorizer_model = CountVectorizer(
    stop_words= get_stopwords(),
    ngram_range=(1, 2),
    min_df=2,
    lowercase=True
)

In the next cell, I run the Bertopic model. I finetune it so that I control more the granularity of the model. I select `n_neighbors` and `min_cluster_size` to be equal to 50. This enables me to get 108 topics with enough granularity. For example, I have two climate topics, one which focuses on financial risks from climate, the other focuses on green transition and ESG. These are not exactly the same and it is interesting to see these differences (see below for this example).

In [8]:
docs = df_chunks_all['chunk_text'].tolist()

RANDOM_SEED = 42
default_umap_parameters = {
    "n_neighbors": 50,
    "n_components": 5,
    "min_dist": 0.0,
    "metric": "cosine",
}
hdbscan_model = HDBSCAN(
    min_cluster_size=50,      # plus petit = plus de topics
    min_samples=5,            # moins strict = moins d'outliers
    metric='euclidean',
    cluster_selection_method='leaf',  # 'leaf' crée plus de petits clusters
    prediction_data=True
)

topic_model = BERTopic(
    language="english",
    embedding_model="hugom123/bge-centralbank",
    vectorizer_model=vectorizer_model,
    umap_model=UMAP(**default_umap_parameters, random_state=42),
    hdbscan_model=hdbscan_model,
    min_topic_size=20,
   nr_topics=None,
    calculate_probabilities=True,
   verbose=True,
)

topics, probs = topic_model.fit_transform(
    documents=docs,
    embeddings=embeddings
)

df_chunks_all['topic']      = topics
df_chunks_all['topic_prob'] = probs.max(axis=1) if probs.ndim > 1 else probs

print(f"\nNombre de topics : {topic_model.get_topic_info().shape[0] - 1}")

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 6386.19it/s]
2026-04-29 19:32:55,321 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-04-29 19:34:28,266 - BERTopic - Dimensionality - Completed ✓
2026-04-29 19:34:28,271 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-04-29 19:34:41,367 - BERTopic - Cluster - Completed ✓
2026-04-29 19:34:41,388 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-04-29 19:34:54,900 - BERTopic - Representation - Completed ✓



Nombre de topics : 109


In [9]:
topics, probabilities = topic_model.transform(documents=docs, embeddings=embeddings)

2026-04-29 19:35:01,583 - BERTopic - Dimensionality - Reducing dimensionality of input embeddings.
2026-04-29 19:35:01,757 - BERTopic - Dimensionality - Completed ✓
2026-04-29 19:35:01,758 - BERTopic - Clustering - Approximating new points with `hdbscan_model`
2026-04-29 19:35:02,886 - BERTopic - Probabilities - Start calculation of probabilities with HDBSCAN
2026-04-29 19:35:18,080 - BERTopic - Probabilities - Completed ✓
2026-04-29 19:35:18,084 - BERTopic - Cluster - Completed ✓


In [10]:
topic_info = topic_model.get_topic_info()
topic_info

,Topic,Count,Name,Representation,Representative_Docs
0,-1,10578,-1_credit_term_risks_us,"[credit, term, risks, us, see, prices, banking...","[Fortunately, this scenario has so far not com..."
1,0,416,0_liquidity_operations_refinancing_refinancing...,"[liquidity, operations, refinancing, refinanci...",[We have decided on three-year refinancing ope...
2,1,409,1_fiscal_pact_countries_public,"[fiscal, pact, countries, public, union, struc...","[Nonetheless, further reforms in the fiscal ar..."
3,2,362,2_supervisory_supervision_eu_banking,"[supervisory, supervision, eu, banking, ssm, e...","[In addition, a new idea of reform contracts w..."
4,3,359,3_medium_medium term_outlook_term,"[medium, medium term, outlook, term, developme...","[Turning to the price developments, annual HIC..."
...,...,...,...,...,...
105,104,52,104_flows_emerging_emes_inflows,"[flows, emerging, emes, inflows, push, pull fa...",[II. Growing challenges in the Current IMFS Ch...
106,105,52,105_productivity_1995_per_output,"[productivity, 1995, per, output, labor produc...",[Different methods of data construction will y...
107,106,52,106_loans_liquidity_lending_credit,"[loans, liquidity, lending, credit, commercial...",[BIS central bankers speeches 3 recognition th...
108,107,51,107_spending_housing_sales_construction,"[spending, housing, sales, construction, house...","[Hence, with production running well below sal..."


The BERTopic model identifies **109 topics** from the corpus of 30,401 chunks. Several of them are immediately interpretable and consistent with what one would expect from central bank communication over the 2001--2023 period:

- **Topic 15 — Housing & Mortgages** (`mortgage, mortgages, subprime, borrowers`):  not surprising given that the 2008 financial crisis shed light on the interaction between the housing market and financial stability; we will examine whether this topic is predominantly concentrated in the post-2008 period.

- **Topic 41 — Digital Money & Payments** (`digital, money, payments, payment`): distinct from the crypto topic, this cluster captures the debate around Central Bank Digital Currencies (CBDCs) and the modernisation of payment systems.

- **Topic 38 — Inequality & Wealth** (`wealth, income, inequality, families`): a particularly interesting topic, as distributional concerns are not historically part of the core central bank mandate. Its presence in the corpus warrants further investigation into its temporal distribution.

- **Topics 77 & 94 — Climate Change** (`climate change, risks, green, transition`): consistent with the findings of Campiglio et al. (2025); BERTopic identifies two distinct climate-related clusters — one centred on climate as a **financial risk** (stress testing, TCFD disclosures) and one on the **green transition** agenda (ESG, net zero, carbon) — which confirms and refines their dictionary-based approach.

- **Topic 90 — Crypto & Stablecoins** (`crypto, stablecoins, stablecoin, bitcoin`): an emerging topic reflecting the growing attention of central bankers to decentralised finance and the regulatory challenges it poses.

- **Topic 91 — 2008 Financial Crisis** (`aig, credit, treasury, institutions`): this topic clearly captures the acute phase of the 2008 crisis, referencing key institutional actors (AIG, Treasury) and emergency interventions.

However, **Topic -1 accounts for 10,578 chunks, representing 34.8% of the corpus** — a substantial share of documents that HDBSCAN was unable to assign to any cluster. This is a known limitation of density-based clustering: documents that do not belong to a sufficiently dense region of the embedding space are labelled as outliers. 

To address this, I apply the `reduce_outliers` method with the **embedding strategy**. This approach assigns each outlier chunk to the topic whose centroid is closest in the embedding space, using cosine similarity. 

In [11]:
new_topics = topic_model.reduce_outliers(
    documents=docs,
    topics=topics,
    probabilities=probs,
    embeddings=embeddings,
    strategy="embeddings",
    threshold=0.0
)

# Mettre à jour le modèle
topic_model.update_topics(
    docs,
    topics=new_topics,
    vectorizer_model=vectorizer_model
)

df_chunks_all['topic'] = new_topics
print(f"Outliers restants : {(pd.Series(new_topics) == -1).sum()}")
print("\n=== Top 30 topics ===")
print(topic_model.get_topic_info().head(30))
print(topic_model.get_topic_info().head(30))

with fs.open("lelkamel/chunks/df_chunks_with_topics.csv", "wb") as f:
    df_chunks_all.to_csv(f, index=False)
print("df_chunks_with_topics sauvegardé sur S3 !")

2026-04-29 19:35:18,650 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure that manually assigning topics is the last step in the pipeline.Note that topic embeddings will also be created through weightedc-TF-IDF embeddings instead of centroid embeddings.


Outliers restants : 0

=== Top 30 topics ===
    Topic  Count                                               Name  \
0       0    497          0_liquidity_operations_refinancing_credit   
1       1    435                     1_fiscal_pact_countries_public   
2       2    428               2_supervisory_supervision_eu_banking   
3       3    515                  3_medium_medium term_term_outlook   
4       4    386  4_labour_productivity_labour productivity_reforms   
5       5    414  5_macroprudential_prudential_macro prudential_...   
6       6    328              6_integration_single_integrated_union   
7       7    290       7_convergence_member states_member_accession   
8       8    264                       8_budget_fiscal_tax_spending   
9       9    464                               9_london_mpc_us_city   
10     10    332                  10_basel_basel ii_ii_requirements   
11     11    326                       11_rule_taylor_models_output   
12     12    254                

I don't have any outlier anymore ! See the next notebook for more visualisation.

In [16]:
#I save the model
topic_model.save(
    "./topic_model_v2",
    serialization="safetensors",
    save_ctfidf=True,
    save_embedding_model=False
)
print("Modèle sauvegardé !")

2026-04-29 19:37:01,383 - BERTopic - WARNING: You are saving a BERTopic model without explicitly defining an embedding model.If you are using a sentence-transformers model or a HuggingFace model supportedby sentence-transformers, please save the model by using a pointer towards that model.For example, `save_embedding_model='sentence-transformers/all-mpnet-base-v2'`


Modèle sauvegardé !


In [17]:

fs = s3fs.S3FileSystem(
    client_kwargs={"endpoint_url": "https://minio.lab.sspcloud.fr"}
)

# Uploader tous les fichiers du dossier
local_dir = "./topic_model_v2"
s3_dir = "lelkamel/chunks/topic_model_v2"

for filename in os.listdir(local_dir):
    local_path = os.path.join(local_dir, filename)
    s3_path = f"{s3_dir}/{filename}"
    fs.put(local_path, s3_path)
    print(f"Uploaded : {filename}")

print("Modèle sauvegardé sur S3 !")

Uploaded : ctfidf.safetensors
Uploaded : config.json
Uploaded : topics.json
Uploaded : topic_embeddings.safetensors
Uploaded : ctfidf_config.json
Modèle sauvegardé sur S3 !


### Visualisation of the results

In [12]:
setup_plotly()

nbformat : 5.10.4


In [13]:
fig = topic_model.visualize_hierarchy()



In [14]:
# Sauvegarder en HTML puis convertir manuellement
fig.write_html("/home/onyxia/work/nlp_central_banks/lyna_work/figures/hierarchy.html")
print("HTML sauvegardé — ouvre le fichier dans le navigateur et fais Ctrl+P → Enregistrer en PDF")

HTML sauvegardé — ouvre le fichier dans le navigateur et fais Ctrl+P → Enregistrer en PDF


You can access the dendogram here : https://nbviewer.org/github/lelkamel/nlp_central_banks/blob/main/lyna_work/figures/hierarchy.html

### Agregation

Now, I will manually merge topics. The hierarchical clustering visualization (see Figure above) and pairwise cosine 
similarities between topic embeddings guide the merging decisions. I adopt a conservative approach: topics are only merged when (i) their cosine similarity exceeds 0.7 and (ii) they are substantively equivalent from an economic standpoint. When two topics are semantically close but economically distinct, I keep them separate.

A first set of merges concerns **duplicate topics** — topics that capture the same concept but in slightly different vocabularies, typically due to institutional differences across central banks. For instance, Topics 24 (`labor_unemployment_workers`) and 25 (`labour_wage_unemployment`) both describe labour market conditions but in American and British English respectively (cosine similarity: 0.909). Similarly, Topics 0, 20 and 21 all relate to liquidity provision and are merged into a single "Monetary Operations & Liquidity" topic (similarity: 0.745).

Another example of a second set of merges concerns **thematically related topics** on unconventional monetary policy. Topics 29 (`reserves_balance sheet`), 42 (`securities_purchase_treasury`) and 73 (`qe_yields_quantitative easing`) all capture the implementation of Quantitative Easing — the large-scale asset purchase programmes introduced by central banks in the aftermath of the 2008 crisis — and are merged accordingly. Two related topics are however kept separate for analytical reasons. Topic 57 (`yields_yield_term_bond`) captures the *market consequences* of QE — yield compression and term premium dynamics — rather than the policy instrument itself, and thus represents the transmission channel of unconventional monetary policy. Topic 74 (`guidance_forward guidance_forward_expectations`) captures forward guidance, a conceptually distinctunconventional tool that operates through the management of expectations about future interest rates rather than through balance sheet expansion. Tracking these three topics separately allows us to distinguish between the *implementation* ofunconventional policies, their *market transmission*, and their *communication* dimension — a distinction that will prove useful in the dynamic topic modeling analysis.

I detail more the merging choices in te appendix of my paper. To see the analysis at the agregate level, skip this part !

In [15]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

def check_similarity(topic_pairs, topic_model):
    topic_embeddings = topic_model.topic_embeddings_
    topic_info = topic_model.get_topic_info()
    
    for pair in topic_pairs:
        print(f"\n{'='*60}")
        for t in pair:
            name = topic_info[topic_info['Topic'] == t]['Name'].values[0]
            print(f"  Topic {t} : {name}")
        
        emb_pair = np.array([topic_embeddings[t+1] for t in pair])
        sim_matrix = cosine_similarity(emb_pair)
        
        # Print pairwise similarities
        for i in range(len(pair)):
            for j in range(i+1, len(pair)):
                sim = sim_matrix[i, j]
                emoji = "✅" if sim > 0.7 else "⚠️" if sim > 0.5 else "❌"
                print(f"\n  {emoji} Topic {pair[i]} vs Topic {pair[j]} : {sim:.3f}")

In [41]:
fig = topic_model.visualize_barchart(
    topics=[26],  # le numéro du topic après merge
    n_words=15,
    title="Top words — Oil, energy, gas"
)
fig.write_html("./figures/barchart_oil_energy_gas.html")

In [38]:
check_similarity([
    [25, 23, 36, 19, 62, 55],  # Theory
    [73, 50, 76, 78, 64, 86],  # Communication
], topic_model)


  Topic 25 : 25_expectations_prices_core_phillips
  Topic 23 : 23_funds_longer_employment_longer run
  Topic 36 : 36_real_equilibrium_low_funds
  Topic 19 : 19_rule_taylor_models_money
  Topic 62 : 62_targeting_target_elb_level
  Topic 55 : 55_target_mpc_targeting_output

  ✅ Topic 25 vs Topic 23 : 0.799

  ❌ Topic 25 vs Topic 36 : 0.036

  ❌ Topic 25 vs Topic 19 : 0.188

  ❌ Topic 25 vs Topic 62 : 0.300

  ❌ Topic 25 vs Topic 55 : 0.117

  ❌ Topic 23 vs Topic 36 : 0.261

  ❌ Topic 23 vs Topic 19 : 0.234

  ❌ Topic 23 vs Topic 62 : 0.336

  ❌ Topic 23 vs Topic 55 : 0.223

  ❌ Topic 36 vs Topic 19 : 0.315

  ❌ Topic 36 vs Topic 62 : 0.273

  ❌ Topic 36 vs Topic 55 : 0.231

  ❌ Topic 19 vs Topic 62 : 0.278

  ❌ Topic 19 vs Topic 55 : 0.274

  ❌ Topic 62 vs Topic 55 : 0.279

  Topic 73 : 73_guidance_forward guidance_forward_expectations
  Topic 50 : 50_communication_decisions_public_expectations
  Topic 76 : 76_transparency_disclosure_information_communication
  Topic 78 : 78_review_goal

In [ ]:
print("=== Top words per theoretical topic ===\n")

theoretical_topics = [19, 36, 62, 55, 66]

for t in theoretical_topics:
    words = topic_model.get_topic(t)[:8]
    word_list = [w for w, _ in words]
    print(f"Topic {t} : {word_list}")
    
    # Also show representative doc
    info = topic_model.get_topic_info()
    rep_docs = info[info['Topic'] == t]['Representative_Docs'].values[0]
    print(f"  → {rep_docs[0][:150]}")
    print()

In [ ]:
check_similarity([[94, 77]], topic_model)

In [ ]:
# Merge all topics
topic_model.merge_topics(docs, [
    [27, 7, 45, 87, 80, 48],          # European union and single market integration
    [14, 108, 6],                       # European governance & integration
    [62, 85],                           # Financial market integration & payments
    [94, 77],                           # Climate change
    [78, 52, 41, 90],                   # Digital currency & payments
    [61, 44, 79],                       # Social & inclusion
    [63, 71],                           # Community development finance
    [88, 56, 37, 23, 36, 58, 10],      # Banking regulation & supervision
    [21, 91, 106, 20],                  # Crisis lending & emergency liquidity
    [81, 16],                           # Resolution & bail-in
    [69, 89, 5],                        # Macroprudential & financial stability
    [83, 99, 101, 76, 72],             # Global governance & globalization
    [22, 104, 95],                      # Global imbalances & capital flows
    [92, 82],                           # Asia & emerging markets
    [31, 8],                            # Fiscal policy & public finance
    [103, 4, 32],                       # Structural reforms
    [24, 25, 105, 53],                  # Labour market & productivity
    [38, 70],                           # Household concerns & inequality
    [40, 50, 15],                       # Housing market & mortgage credit
    [35, 26],                           # Energy & commodity prices
    [97, 12],                           # Asset prices & bubbles
    [84, 60, 51, 107, 98, 9, 34, 3],   # Macroeconomic outlook
    [29, 42, 73],                       # QE & asset purchases
    [64, 96],                           # Strategy & longer-run framework
    [54, 39, 49],                       # Historical & ceremonial
    [66, 28],                           # Communication & transparency
    [75, 100, 33, 11, 43, 102, 65, 67] # Theoretical & technical
])

In [47]:
df_chunks_all['topic'] = topic_model.topics_

print(f"Topics after merge : {topic_model.get_topic_info().shape[0] - 1}")
print(topic_model.get_topic_info()[['Topic', 'Count', 'Name']].to_string())

Topics after merge : 43
    Topic  Count                                               Name
0       0   2521                   0_prices_demand_outlook_recovery
1       1   1979             1_basel_regulatory_banking_supervision
2       2   1526             2_expectations_target_term_uncertainty
3       3   1521                    3_union_countries_europe_single
4       4   1154                   4_liquidity_credit_funding_funds
5       5   1084                    5_integration_union_europe_ecbs
6       6    982          6_productivity_labor_unemployment_workers
7       7    759           7_labour_reforms_productivity_structural
8       8    738           8_trade_countries_globalisation_exchange
9       9    706          9_macroprudential_stress_macro_prudential
10     10    650                10_mortgage_housing_mortgages_house
11     11    621                   11_money_great_century_economics
12     12    575       12_reserves_securities_balance_balance sheet
13     13    572        

In [49]:
topic_names = {
    0:  "Macroeconomic Outlook & Assessment",        # prices_demand_outlook_recovery
    1:  "Banking Regulation & Supervision",           # basel_regulatory_banking_supervision
    2:  "Theoretical & Technical Speeches",           # expectations_target_term_uncertainty
    3:  "European Integration & Single Market",       # union_countries_europe_single
    4:  "Crisis Lending & Emergency Liquidity",       # liquidity_credit_funding_funds
    5:  "European Governance & Institutional",        # integration_union_europe_ecbs
    6:  "Labour Market & Employment",                 # productivity_labor_unemployment_workers
    7:  "Structural Reforms",                         # labour_reforms_productivity_structural
    8:  "Globalization & Trade",                      # trade_countries_globalisation_exchange
    9:  "Macroprudential & Financial Stability",      # macroprudential_stress_macro_prudential
    10: "Housing Market & Mortgage Credit",           # mortgage_housing_mortgages_house
    11: "Historical & Ceremonial Speeches",           # money_great_century_economics
    12: "Quantitative Easing & Asset Purchases",      # reserves_securities_balance_balance sheet
    13: "Digital Finance, Payments & Crypto",         # payments_payment_digital_money
    14: "Global Imbalances & Capital Flows",          # foreign_account_flows_dollar
    15: "ECB Monetary Operations & Liquidity",        # liquidity_operations_credit_refinancing
    16: "Resolution & Bank Failure",                  # resolution_creditors_firms_banking
    17: "Financial Crises",                           # credit_crises_finance_sector
    18: "European Fiscal Framework (SGP)",            # fiscal_countries_pact_public
    19: "Banking Supervision (SSM/EU)",               # supervisory_supervision_banking_eu
    20: "Longer-Run Strategy & Fed Framework",        # funds_employment_longer_maximum
    21: "Asset Prices & Bubbles",                     # prices_bubble_bubbles_investors
    22: "Energy & Commodity Prices",                  # oil_prices_energy_gas
    23: "Fiscal Policy & Public Finance",             # budget_fiscal_spending_tax
    24: "Communication & Transparency",               # communication_transparency_public_decisions
    25: "Social Outreach & Financial Inclusion",      # education_students_women_school
    26: "Household Concerns & Inequality",            # income_wealth_households_inequality
    27: "Derivatives & Counterparty Risk",            # derivatives_credit_hedge_counterparty
    28: "Banking Union & Resolution Framework",       # banking_union_banking union_resolution
    29: "R-star & Equilibrium Real Rate",             # real_equilibrium_low_funds
    30: "Climate Change",                             # climate_climate change_transition_change
    31: "Bank Profitability",                         # profitability_banking_sector_banking sector
    32: "Credit Risk Management",                     # management_credit_risks_institutions
    33: "Financial Market Integration",               # integration_sepa_securities_bond
    34: "Community Development Finance",              # businesses_development_business_credit
    35: "Central Bank Independence & Governance",     # independence_decisions_mpc_members
    36: "Bond Markets & Yield Compression",           # yields_term_yield_bond
    37: "Asia & Emerging Markets",                    # asian_asia_trade_emerging
    38: "Forward Guidance",                           # guidance_forward guidance_forward_expectations
    39: "Community Banking (Fed)",                    # banking_rural_local_bankers
    40: "Statistics & Data Methodology",              # statistics_data_statistical_information
    41: "Shadow Banking",                             # shadow_shadow banking_banking_sector
    42: "Insurance & Solvency II",                    # insurance_insurers_solvency_solvency ii
    43: "LIBOR Transition"                            # libor_sonia_transition_sofr
}

# Apply to dataframe
df_chunks_all['topic_name'] = df_chunks_all['topic'].map(topic_names)

# Verify
print(df_chunks_all.groupby(['topic', 'topic_name']).size()
      .reset_index(name='count')
      .to_string())

# Save updated dataframe
with fs.open("lelkamel/chunks/df_chunks_final_topics.csv", "wb") as f:
    df_chunks_all.to_csv(f, index=False)
print("Saved !")

    topic                              topic_name  count
0       0      Macroeconomic Outlook & Assessment   2521
1       1        Banking Regulation & Supervision   1979
2       2        Theoretical & Technical Speeches   1526
3       3    European Integration & Single Market   1521
4       4    Crisis Lending & Emergency Liquidity   1154
5       5     European Governance & Institutional   1084
6       6              Labour Market & Employment    982
7       7                      Structural Reforms    759
8       8                   Globalization & Trade    738
9       9   Macroprudential & Financial Stability    706
10     10        Housing Market & Mortgage Credit    650
11     11        Historical & Ceremonial Speeches    621
12     12   Quantitative Easing & Asset Purchases    575
13     13      Digital Finance, Payments & Crypto    572
14     14       Global Imbalances & Capital Flows    508
15     15     ECB Monetary Operations & Liquidity    497
16     16               Resolut

In [48]:
# Save df_chunks_all with final topics
with fs.open("lelkamel/chunks/df_chunks_final_topics.csv", "wb") as f:
    df_chunks_all.to_csv(f, index=False)
print(f"df_chunks_final_topics sauvegardé : {df_chunks_all.shape}")

df_chunks_final_topics sauvegardé : (23913, 12)


In [50]:
# Get topics with more than 500 chunks
topic_info = topic_model.get_topic_info()
big_topics = topic_info[topic_info['Count'] > 500]['Topic'].tolist()
print(f"Topics with > 500 chunks : {big_topics}")

# Visualise
fig = topic_model.visualize_barchart(
    topics=big_topics,
    n_words=10,
    height=300,
)

# Save to HTML
fig.write_html("/home/onyxia/work/nlp_central_banks/lyna_work/figures/barchart_big_topics.html")
print("Saved !")

Topics with > 500 chunks : [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14]
Saved !
